# Ce que produisent huit lignes de Python · *What eight lines of Python produce*

Notebook compagnon du chapitre **19. Premier script Python : charger une série FRED et la tracer en dix lignes** — [lire l'article](https://nmlab.io/ressources/premier-script-python-fred).
Companion notebook to chapter **19. Your First Python Script: Load a FRED Series and Plot It in Ten Lines** — [read the article](https://nmlab.io/en/ressources/first-python-script-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_gdp() -> Series:
    """PIB réel des États-Unis (GDPC1), en direct depuis FRED / U.S. real GDP, live from FRED."""
    return nm.load_fred("GDPC1")

gdp = load_gdp()


import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Ce que produisent huit lignes de Python",
        sub="Le PIB réel des États-Unis (GDPC1), chargé depuis FRED et tracé",
        ylab="milliards de dollars de 2017"),
    "en": dict(
        title="What eight lines of Python produce",
        sub="U.S. real GDP (GDPC1), loaded from FRED and plotted",
        ylab="billions of 2017 dollars"),
}

def caption(gdp: Series, lang: str) -> str:
    """Note dynamique : le dernier point de PIB / dynamic note: the latest GDP point."""
    latest, when = gdp.iloc[-1], gdp.index[-1]
    quarter = (when.month - 1) // 3 + 1
    if lang == "fr":
        value = f"{latest:,.0f}".replace(",", " ")
        ordinal = "1ᵉʳ" if quarter == 1 else f"{quarter}ᵉ"
        return ("Aucun fichier à télécharger à la main : le script va chercher la série directement sur FRED et la dessine.\n"
                f"Dernier point : {value} milliards au {ordinal} trimestre {when.year}. Source : BEA via FRED (GDPC1).")
    return ("Nothing to download by hand: the script fetches the series straight from FRED and draws it.\n"
            f"Latest point: ${latest:,.0f} billion in Q{quarter} {when.year}. Source: BEA via FRED (GDPC1).")

def build_figure(gdp: Series, lang: str) -> Figure:
    """PIB réel en aire remplie, sur toute la période disponible."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1026)
    ax = nm.axes(fig, left=0.12)
    ax.fill_between(gdp.index, gdp, color=nm.COLORS["blue"], alpha=0.14)
    ax.plot(gdp.index, gdp, color=nm.COLORS["blue"], linewidth=2.9)
    ax.set_ylim(0, 26_000)
    ax.set_yticks(range(0, 26_000, 5000))
    ax.yaxis.set_major_formatter(nm.thousands(lang))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, caption(gdp, lang))
    return fig

build_figure(gdp, LANG)